In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
import pandas as pd
import rubin_sim
import rubin_sim.maf as maf
#from rubin_sim.data import get_baseline
print(rubin_sim.__version__, 'yo')
from os.path import splitext, basename
from rubin_scheduler.scheduler.utils import SkyAreaGenerator

0.1.dev1813+gf6a2d0c yo


In [20]:
import warnings

import healpy as hp
import numpy as np
from scipy import interpolate
from scipy.stats import median_abs_deviation
from astropy import units as u
from astropy.coordinates import SkyCoord
from sklearn.cluster import KMeans
from copy import deepcopy

from rubin_sim.maf.maf_contrib.static_probes_fom_summary_metric import StaticProbesFoMEmulatorMetric
from rubin_sim.maf.metrics.area_summary_metrics import AreaThresholdMetric
from rubin_sim.maf.metrics.uniformity_metrics import MultibandExgalM5
from rubin_sim.maf.metrics.base_metric import BaseMetric
from rubin_sim.maf.metrics.simple_metrics import RmsMetric, CountMetric

class MultibandMeanzBiasMetricc(BaseMetric):
    """
    Run as summary metric on MultibandExgalM5.

    Parameters
    ----------
    year : `int`, optional
        The year of the survey to calculate the bias.
        This is used to derive the dm/dz derivative used to translate m5 rms into dz rms.

    meanz_tomograph_model : `dict`
        dictionary containing models calculated for fiducial N(z):

        meanz: numpy.float 
            the meanz within a tomographic bin at a given band
        dz_dm5: numpy.float
            the absolute value of the derivative of z in a bin as a function 
            of the depth, m5 magnitude. This is later used to translate fluctuations
            in depth to fluctuations in tomographic redshift

    Returns
    -------
    result : `float` array
        The ratio of this bias to the desired DESC y1 upper bound on the bias, and the ratio
        between the clbias and the y10 DESC SRD requirement.
        Desired values are less than 1 by Y10.

    Notes
    -----
    This is a summary metric to be run on the results
    of the MultibandExgalM5.

    MultibandExgalM5 provides the m5 depth in all LSST bands given a specific slice.

    This summary metric takes those depths and reads the derivatives from the tomogrpahic model, 
    computes the bias in shear signal Cl and then computes the ratio of that bias to the Y1 goal and
    Y10 science requirement. 
    """

    def __init__(self, 
                 meanz_tomography_model,
                 filter_list=["u", "g", "r", "i", "z", "y"], 
                 year=10, n_filters=6, **kwargs):

        super().__init__(col="metricdata",**kwargs)
        # Set mask_val, so that we receive metric_values.filled(mask_val)
        self.mask_val = hp.UNSEEN
        self.badval = hp.UNSEEN
        self.rmsMetric = RmsMetric()
        self.year = year
        self.filter_list = filter_list
        

    def run(self, data_slice, slice_point=None):

        def compute_dzfromdm(zbins, model_z, band_ind, year):

            """This computes the dm/dz relationship calibrated from simulations
            by Jeff Newmann and Qianjun Hang, which forms the meanz_tomographic_model.

            Parameters
            ----------
            zbins : `int`
                The number of tomographic bins considered. For now this is zbins < 5
            filter : `str`
                The assumed filter band
            model_z: dict
                The meanz_tomographic_model assumed in this work

            Returns
            -------
            dzdminterp : `float` array
                The interpolated value of the derivative dz/dm
            meanzinterp : `float` array
                The meanz in each tomographic bin.

            """
            import pandas as pd

            filter_list = ["u", "g", "r", "i", "z", "y"]
            meanzinterp=np.zeros(zbins)
            dzdminterp=np.zeros(zbins)

            for z in range(zbins):
                meanzinterp[z]=model_z["year%i"%(year+1)]["meanz"][z][filter]
                dzdminterp[z] =model_z["year%i"%(year+1)]["dz_dm5"][z][filter]

            return dzdminterp, meanzinterp

        ## Not entirely sure if we should include these here or in a separate module/file
        def use_zbins(meanz_vals, figure_9_mean_z=np.array([0.2, 0.4, 0.7, 1.0]), figure_9_width=0.2):
            """This computes which redshift bands are within the range
            specified in https://arxiv.org/pdf/2305.15406.pdf and can safely be used
            to compute what Cl bias result from z fluctuations caused by rms variations in the m5.


            Parameters
            ----------
            meanz_vals : `float` array
                Array of meanz values to be used.

            Returns
            -------
            use_bins : `boolean` array
                An array of boolean values of length meanz_vals

            """
            max_z_use = np.max(figure_9_mean_z) + 2 * figure_9_width
            use_bins = meanz_vals < max_z_use

            return use_bins

        def compute_Clbias(meanz_vals, scatter_mean_z_values):
            """This computes the Cl bias
            that results z fluctuations caused by rms variations in the m5.



            Parameters
            ----------
            meanz_vals : `float` array
                Array of meanz values to be used.

            scatter_mean_z_values : `float` array
                Array of rms values of the z fluctuations


            Returns
            -------
            clbiasvals : `float` array
                An array of values of the clbias

            mean_z_values_use :  `float` array
                An array of the meanz values that are within the interpolation range of 2305.15406

            Notes
            ------
            This interpolates from the Figure 9 in https://arxiv.org/pdf/2305.15406.pdf

            """
            import numpy as np

            figure_9_mean_z = np.array([0.2, 0.4, 0.7, 1.0])
            figure_9_Clbias = np.array([1e-3, 2e-3, 5e-3, 1.1e-2])
            figure_9_width = 0.2
            figure_9_mean_z_scatter = 0.02

            mzvals = np.array([float(mz) for mz in meanz_vals])
            sctz = np.array([float(sz) for sz in scatter_mean_z_values])

            fit_res = np.polyfit(figure_9_mean_z, figure_9_Clbias, 2)
            poly_fit = np.poly1d(fit_res)
            use_bins = use_zbins(meanz_vals, figure_9_mean_z, figure_9_width)

            mean_z_values_use = mzvals[use_bins]
            sctz_use = sctz[use_bins]

            Clbias = poly_fit(mean_z_values_use)
            rescale_fac = sctz_use / figure_9_mean_z_scatter
            Clbias *= rescale_fac
            fit_res_bias = np.polyfit(mean_z_values_use, Clbias, 1)
            poly_fit_bias = np.poly1d(fit_res_bias)

            clbiasvals = poly_fit_bias(mean_z_values_use)
            return clbiasvals, mean_z_values_use

        
        result = np.empty(1, dtype=[("name", np.str_, 20), ("y1ratio", float), ("y10ratio", float)])
        result["name"][0] = "MultibandMeanzBiasMetricc"
        #print(result["name"], 'argh')

        # Technically don't need this for now (isn't used in previous one)
        # need to define an array of bad values for the masked pixels
        badval_arr = np.repeat(self.badval, len(self.filter_list))
        
        print(data_slice["metricdata"], 'wut')
        # converts the input recarray to an array
        data_slice_list = [
            badval_arr if isinstance(x, float) else x for x in data_slice["metricdata"].tolist()
        ]
        
        lengths = np.array([len(x) for x in data_slice_list])
    

        data_slice_arr = np.asarray(data_slice_list, dtype=float).T
        
        print( data_slice_arr, 'what is up')

        data_slice_arr[~np.isfinite(data_slice_arr)] = (
            hp.UNSEEN
        )  # need to work with TotalPowerMetric and healpix
        

        # Kulge to get things going
        data_slice_arr = data_slice_arr/data_slice_arr
        data_slice_arr = np.random.rand(len(data_slice_arr))
        
        # measure rms in each bin.
        # The original metric returns an array at each slice_point (of the
        # original slicer) -- so there is a bit of "rearrangement" that
        # has to happen to be able to pass a np.array with right dtype
        # (i.e. dtype = [("metricdata", float)]) to each call to
        # the rmsMetric `run` methods.

        #test = self.rmsMetric.run(np.core.records.fromrecords(0.341, dtype=[("metricdata", float)]))
        #print(test) 

        rmsarray = np.array(
            [
                np.std(np.array(x[x != hp.UNSEEN]))
                for x in data_slice_arr
            ]
        )

        # rmsarray = np.array(
        #     [
        #         self.rmsMetric.run(np.core.records.fromrecords(np.array(x), dtype=[("metricdata", float)]))
        #         # self.rmsMetric.run(np.core.records.fromrecords(x, dtype=[("metricdata", float)]))
        #         for x in data_slice_arr
        #     ]
        # )  # rms values

        print(rmsarray, 'huh')
        totdz = 0
        avmeanz = 0
        totclbias = 0

        for i, filt in enumerate(self.filter_list):
            dzdminterp, meanzinterp = compute_dzfromdm(self.zbins, filt, self.year)
            print(dzdminterp, 'hi renee')
            stdz = [float(np.abs(dz)) * float(rmsarray[i]) for dz in dzdminterp]

            clbias, meanz_use = compute_Clbias(meanzinterp, stdz)

            totdz += [float(st ** 2) for st in stdz]
            totclbias += clbias
            avmeanz += meanzinterp

        # These should be included in self rather than hard coded
        y10_req = 0.003
        y1_goal = 0.013

        # clbiastot = np.max(clbias) # if adding doesn't work over z range -- CHECK
        # y10ratio = totclbias / y10_req
        # y1ratio = totclbias / y1_goal

        y10ratio = 3 / y10_req
        y1ratio = 3 / y1_goal

        print('hi renee' )
        print(y10ratio)
        result["y1ratio"] = y1ratio
        result["y10ratio"] = y10ratio
        return y10ratio

In [21]:
# fixing a weird issue
import os
os.environ['RUBIN_SIM_DATA_DIR'] = '/pscratch/sd/r/rhlozek/rubin_sim_data'
os.environ['RUBIN_SIM_DATA_DIR'] 

'/pscratch/sd/r/rhlozek/rubin_sim_data'

# Demo of meanz bias Metric

In [22]:
#from rubin_sim.maf.metrics.cosmology_summary_metrics import MultibandMeanzBiasMetric

from rubin_sim.maf.metrics.uniformity_metrics import MultibandExgalM5

from rubin_sim.maf.metrics.tomography_models import DENSITY_TOMOGRAPHY_MODEL
from rubin_sim.maf.metrics.tomography_models import MEANZ_TOMOGRAPHY_MODEL

# These have all the model information in it so that we don't have to load pkl files from the density computations


In [ ]:
# ## MEANZ demo from Boris adapted

# nside = 32

# sim_list = [
#           '/pscratch/sd/r/rhlozek/rubin_sim_data/sim_baseline/baseline_v4.0_10yrs.db',
#           '/pscratch/sd/r/rhlozek/rubin_sim_data/sim_baseline/baseline_v3.6_10yrs.db',
#           '/pscratch/sd/r/rhlozek/rubin_sim_data/v3.4/sim_baseline/baseline_v3.4_10yrs.db',
#           '/pscratch/sd/r/rhlozek/rubin_sim_data/v3.4/uniform_roll/roll_uniform_mjdp0_v3.4_10yrs.db',
#           '/pscratch/sd/r/rhlozek/rubin_sim_data/noroll/noroll_v3.6_10yrs.db',
#         '/pscratch/sd/r/rhlozek/rubin_sim_data/four_cycle_2/four_cycle_2_v3.6_10yrs.db']
# name_list = [splitext(basename(sim))[0] for sim in sim_list]

# years = range(1, 2) 
# surveyAreas = SkyAreaGenerator(nside=nside)
# map_footprints, map_labels = surveyAreas.return_maps()
# slicer = maf.HealpixSubsetSlicer(
#     nside=nside, 
#     hpid=np.where(map_labels == "lowdust")[0],#hpid=np.arange(hp.nside2npix(nside)), 
#     use_cache=False)

# results_allruns= {}
# for opsim_fname, run_name in zip(sim_list, name_list):
#     if '3.4' in opsim_fname:
#         extra_string = ''
#     else:
#         extra_string = 'scheduler_'

#     results_allyears = np.zeros((len(years), ))
#     # loop over years 
#     for iy, year in enumerate(years):
#         print('year hi hi', year)

#         days = year*365.25
#         constraint_str = extra_string+'note not like "DD%" and night <= XX and '+extra_string+'note not like "twilight_near_sun" '
#         constraint_str = constraint_str.replace('XX','%d'%days)

        
#         metric = MultibandExgalM5() 
#         summary_metrics = [MultibandMeanzBiasMetricc(MEANZ_TOMOGRAPHY_MODEL['year'+str(year)],
#                                                     year=year, 
#                                                     metric_name='MultibandMeanzBias')]
#         depth_map_bundles = [maf.MetricBundle(
#             metric=metric,
#             slicer=slicer,
#             constraint=constraint_str,
#             run_name=run_name,
#             summary_metrics=summary_metrics
#         )]
#         bd = maf.metricBundles.make_bundles_dict_from_list(depth_map_bundles)
#         bgroup = maf.MetricBundleGroup(bd, opsim_fname)
#         bgroup.run_all()
#         results_allyears[iy] = bd[list(bd.keys())[0]].summary_values['MultibandMeanzBias']
#         print(results_allyears[iy], iy, 'hi')
        
#     results_allruns[run_name] = results_allyears

    

In [23]:
nside = 32
sim_list = [
          '/pscratch/sd/r/rhlozek/rubin_sim_data/sim_baseline/baseline_v4.0_10yrs.db',
          '/pscratch/sd/r/rhlozek/rubin_sim_data/v3.4/sim_baseline/baseline_v3.4_10yrs.db',
          '/pscratch/sd/r/rhlozek/rubin_sim_data/noroll/noroll_v3.6_10yrs.db',
                    '/pscratch/sd/r/rhlozek/rubin_sim_data/v3.4/uniform_roll/roll_uniform_mjdp0_v3.4_10yrs.db',
        '/pscratch/sd/r/rhlozek/rubin_sim_data/four_cycle_2/four_cycle_2_v3.6_10yrs.db']
name_list = [splitext(basename(sim))[0] for sim in sim_list]
#7C9EB2 = baseline 3.4
colors = ['#000000','#7C9EB2',  '#7C9EB2','#FFB400','#6f6fed'] 

#7C9EB2 = baseline 3.4
#7C9EB2 (dashed) = baseline 3.4
#6f6fed = baseline 4.0 four cycle
#FFB400 = uniform 3.4
#000000 = baseline 4.0

years = range(1, 11) 
surveyAreas = SkyAreaGenerator(nside=nside)
map_footprints, map_labels = surveyAreas.return_maps()
slicer = maf.HealpixSubsetSlicer(
    nside=nside, 
    hpid=np.where(map_labels == "lowdust")[0],#hpid=np.arange(hp.nside2npix(nside)), 
    use_cache=False)

results_allruns= {}
for opsim_fname, run_name in zip(sim_list, name_list):
    if '3.4' in opsim_fname:
        extra_string = ''
    else:
        extra_string = 'scheduler_'

    results_allyears = np.zeros((len(years), ))
    # loop over years 
    for iy, year in enumerate(years):
        print('year', year)

        days = year*365.25
        constraint_str = extra_string+'note not like "DD%" and night <= XX and '+extra_string+'note not like "twilight_near_sun" '
        constraint_str = constraint_str.replace('XX','%d'%days)

        metric = MultibandExgalM5() 
        summary_metrics = [MultibandMeanzBiasMetricc(MEANZ_TOMOGRAPHY_MODEL['year'+str(year)],year=year, 
                                                    metric_name='MultibandMeanzBias')]
        depth_map_bundles = [maf.MetricBundle(
            metric=metric,
            slicer=slicer,
            constraint=constraint_str,
            run_name=run_name,
            summary_metrics=summary_metrics
        )]
        
        bd = maf.metricBundles.make_bundles_dict_from_list(depth_map_bundles)
        bgroup = maf.MetricBundleGroup(bd, opsim_fname)
        bgroup.run_all()
        results_allyears[iy] = bd[list(bd.keys())[0]].summary_values['MultibandMeanzBias']
        
    results_allruns[run_name] = results_allyears

Healpix slicer using NSIDE=32, approximate resolution 109.935565 arcminutes
year 1


/pscratch/sd/r/rhlozek/rubin_sim/rubin_sim/maf/maps/dust_map.py:46: UserWarning: Slicer value of nside 32 different from map value 128, using slicer value
  warnings.warn(


[-1.6375e+30 -1.6375e+30 -1.6375e+30 ... -1.6375e+30 -1.6375e+30
 -1.6375e+30] wut
[[-1.6375e+30 -1.6375e+30 -1.6375e+30 ... -1.6375e+30 -1.6375e+30
  -1.6375e+30]
 [-1.6375e+30 -1.6375e+30 -1.6375e+30 ... -1.6375e+30 -1.6375e+30
  -1.6375e+30]
 [-1.6375e+30 -1.6375e+30 -1.6375e+30 ... -1.6375e+30 -1.6375e+30
  -1.6375e+30]
 [-1.6375e+30 -1.6375e+30 -1.6375e+30 ... -1.6375e+30 -1.6375e+30
  -1.6375e+30]
 [-1.6375e+30 -1.6375e+30 -1.6375e+30 ... -1.6375e+30 -1.6375e+30
  -1.6375e+30]
 [-1.6375e+30 -1.6375e+30 -1.6375e+30 ... -1.6375e+30 -1.6375e+30
  -1.6375e+30]] what is up
[0. 0. 0. 0. 0. 0.] huh


AttributeError: 'MultibandMeanzBiasMetricc' object has no attribute 'zbins'

In [14]:
year=1
MultibandMeanzBiasMetric(MEANZ_TOMOGRAPHY_MODEL['year'+str(year)],
                                                    year=year, 
                                                    metric_name='MultibandMeanzBias')

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(7, 7), sharex=True)

colors = ['orange', 'blue', 'black', 'red']
for i, run_name in enumerate(results_sigma8_squared_bias.keys()):
    axs[0].plot(years, results_sigma8_squared_bias[run_name], label=run_name, marker='o', color=colors[i])
axs[0].legend()

results_sigma8_squared_bias.keys()
axs[1].set_xlabel('Years')
axs[0].set_ylabel('Bias in sigma8 in units of sigmas')

run_name_ = list(results_sigma8_squared_bias.keys())[0]
for i, run_name in enumerate(results_sigma8_squared_bias.keys()):
    axs[1].plot(years, np.array(years)*0, ls='--', c='orange')
    if run_name != run_name_:
        axs[1].plot(years, results_sigma8_squared_bias[run_name]-results_sigma8_squared_bias[run_name_], label=run_name, marker='o', color=colors[i])
axs[1].set_ylabel('Top panel minus '+run_name_)

In [ ]:
stop

In [ ]:
# sim_list = [
#    '/pscratch/sd/r/rhlozek/rubin_sim_data/v3.6/noroll/noroll_v3.6_10yrs.db',
#    '/pscratch/sd/r/rhlozek/rubin_sim_data/v3.6/sim_baseline/baseline_v3.6_10yrs.db',
#    '/pscratch/sd/r/rhlozek/rubin_sim_data/v3.6/four_cycle_2/four_cycle_2_v3.6_10yrs.db',
# '/pscratch/sd/r/rhlozek/rubin_sim_data/v3.4/sim_baseline/baseline_v3.4_10yrs.db']
# name_list = [splitext(basename(sim))[0] for sim in sim_list]

# years = range(1, 11)  # [1, 2, 4, 7, 10]#


# # large mag_range_tolerated and no min depth in order to make comparison fair between strategies etc
# results_fsky = {}
# results_y1 = {}
# results_y10={}

# for opsim_fname, run_name in zip(sim_list, name_list):
    
#    print('run_name:', run_name)
#     # constraints
#    days = year*365.25
#    constraint_str = 'scheduler_note not like "DD%" and night <= XX and scheduler_note not like "twilight_near_sun" '
#    constraint_str = constraint_str.replace('XX','%d'%days)

#     # results_y1[run_name], _ = extract_meanz_metric(
#     #     opsim_fname, run_name,
#     #     years, 
#     #     MEANZ_TOMOGRAPHY_MODEL,
#     #     nside=64,
#     #     n_filters=6, extinction_cut=0.2
#     # )
    
#    metric = MultibandMeanzBiasMetric(
#          meanz_tomography_model['year'+str(year)], 
#          n_filters=n_filters # cuts going into ExgalM5WithCuts
#         )
    

In [ ]:
# # a simple wrapper around the metrics, to store the results, but not critically needed
# def extract_meanz_metric(
#     opsim_fname, run_fname,
#     years, 
#     meanz_tomography_model, 
#     n_filters = 6, 
#     extinction_cut = 0.2, # sky cuts
#     nside=32
# ):
#     surveyAreas = SkyAreaGenerator(nside=nside)
#     map_footprints, map_labels = surveyAreas.return_maps()
#     slicer = maf.HealpixSubsetSlicer(
#         nside=nside, 
#         hpid=np.where(map_labels == "lowdust")[0],#hpid=np.arange(hp.nside2npix(nside)), 
#         use_cache=False)
    
#     # prepare empty arrays to fill in the results
#     n_bins = 5 # set to 5
#     results_y1 = np.zeros(len(years))
#     results_y10 = np.zeros(len(years))
#     # loop over years
#     all_depth_map_bundles = []
#     for iy, year in enumerate(years):
#         print('year', year)
        
#         # constraints
#         days = year*365.25
#         constraint_str = 'scheduler_note not like "DD%" and night <= XX and scheduler_note not like "twilight_near_sun" '
#         constraint_str = constraint_str.replace('XX','%d'%days)
    
#         # leave empty if not specified
#         mean_depth = {}
#         min_depth_cut = {} 
#         max_depth_cut = {}
       
#         ##############################
#         # now converts depth fluctuations to density fluctuations
#         ##############################
#         metric = MultibandMeanzBiasMetric(
#             meanz_tomography_model['year'+str(year)], 
#             n_filters=n_filters # cuts going into ExgalM5WithCuts
#         )
#         # summary metric measures total power via angular power spectra of healpix map (thus needs nside)
#         # _but_ has a bin-dependent lmax to consider same scales to consider the same scales as a fct of redshift
#         summary_metrics = [
#            MultibandMeanzBiasMetric(
#                 meanz_tomography_model['year'+str(year)]),
#         ]

#         # then standard way of packing MetricBundles into a MetricBundleGroup
#         depth_map_bundles = [maf.MetricBundle(
#             metric=metric,
#             slicer=slicer,
#             constraint=constraint_str,
#             run_name=run_name,
#             summary_metrics=summary_metrics
#         )]
#         bd = maf.metricBundles.make_bundles_dict_from_list(depth_map_bundles)
#         bgroup = maf.MetricBundleGroup(bd, opsim_fname)
#         bgroup.run_all()

#         # compute bias
#         # should probably also return fsky
#         results_y1[iy] = depth_map_bundles[0].summary_values['y1ratio']
#         results_y10[iy] = depth_map_bundles[0].summary_values['y10ratio']
#         all_depth_map_bundles.append(depth_map_bundles[0])

#     return results_y1, results_y10, all_depth_map_bundles